# 03 - Land Cover Classification

**Goal**: Train a Random Forest classifier on CORINE Land Cover labels and produce
a classified land cover map for Lanzarote.

**Workflow**:
1. Load CORINE 2018 labels from GEE → remap 44 classes → our 6
2. Stack 2023 composite bands + 7 spectral indices as features
3. Sample labelled pixels (stratified by class)
4. Train & evaluate **scikit-learn** Random Forest (proper accuracy metrics)
5. Train **GEE** Random Forest → classify full image server-side
6. Visualise the classified map

**Why two classifiers?**  
scikit-learn gives us rigorous metrics (confusion matrix, Kappa, F1 per class) and a
saveable `.joblib` model. The GEE classifier handles the full raster - applying the
model to every pixel across the whole island without downloading gigabytes.

**Phase**: 2 - Classification  
**Labels**: CORINE Land Cover 2018 (Copernicus/EEA), 100m resolution  
**Features**: 6 Landsat bands + 7 spectral indices = 13 features per pixel


In [1]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import ee
import folium
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, cohen_kappa_score,
)

from pipeline.config import (
    CLASS_NAMES, CLASS_COLORS, CORINE_REMAP, N_CLASSES,
)
from pipeline.indices import gee_add_indices

print('All libraries loaded OK')


All libraries loaded OK


In [2]:
GEE_PROJECT = 'project-4cb2ec4e-f113-48f1-8b5'
AOI_COORDS  = [-13.92, 28.80, -13.30, 29.30]
MAP_CENTRE  = [29.05, -13.61]
MAP_ZOOM    = 10

FEATURE_BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2',
                 'ndvi', 'ndwi', 'ndbi', 'savi', 'bsi', 'evi', 'mndwi']

CLASS_PALETTE = [CLASS_COLORS[n] for n in CLASS_NAMES]

ee.Initialize(project=GEE_PROJECT)
aoi = ee.Geometry.Rectangle(AOI_COORDS)
print('GEE initialised')
print(f'Classes: {CLASS_NAMES}')
print(f'Palette: {CLASS_PALETTE}')


GEE initialised
Classes: ['Urban/Built-up', 'Forest/Woodland', 'Water/Wetland', 'Agriculture', 'Barren/Volcanic', 'Shrubland/Matorral']
Palette: ['#E8443A', '#2D8C3C', '#3B82F6', '#F5C542', '#4A4A4A', '#C4A86B']


## 1. Load CORINE Land Cover Labels

CORINE 2018 is available directly in GEE. We remap its 44 classes to our 6
using `config.CORINE_REMAP`.

**Resolution note**: CORINE is 100m, Landsat is 30m. We sample training pixels at
100m scale to match CORINE's resolution and avoid mixed-boundary noise. The trained
classifier then predicts at 30m.


In [ ]:
# Load CORINE 2018 from GEE catalog
corine_raw = ee.Image('COPERNICUS/CORINE/V20/100m/2018').select('landcover').clip(aoi)

# Remap 44 CORINE classes to our 6 using config.CORINE_REMAP
from_vals = list(CORINE_REMAP.keys())
to_vals   = list(CORINE_REMAP.values())

corine_remapped = (
    corine_raw
    .remap(from_vals, to_vals, defaultValue=-1)
    .rename('label')
)

# Mask pixels with no CORINE label (ocean excluded via config - code 523 removed)
corine_labels = corine_remapped.updateMask(corine_remapped.gte(0))

# Count pixels per class - determines which classes we can actually train on
print('Pixel counts per class at 100m scale:\n')
pixel_counts = {}
for i, name in enumerate(CLASS_NAMES):
    count = corine_labels.eq(i).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=100,
        maxPixels=1e8,
    ).getInfo()['label']
    pixel_counts[i] = int(count)
    status = 'OK' if count > 0 else 'NO DATA'
    print(f'  {i} - {name:<22}  {int(count):>6} pixels  [{status}]')

available_classes = [i for i, c in pixel_counts.items() if c > 0]
print(f'\nTrainable classes: {[CLASS_NAMES[i] for i in available_classes]}')
if len(available_classes) < N_CLASSES:
    missing = [CLASS_NAMES[i] for i in range(N_CLASSES) if i not in available_classes]
    print(f'No CORINE data for: {missing}')
    print('These classes will be excluded from training for this AOI.')
    print('ESA WorldCover (10m, global) would provide better coverage.')


In [4]:
# Visualise CORINE labels on a map
def add_ee_layer(fmap, image, vis, name, shown=True, opacity=0.85):
    mid = ee.Image(image).getMapId(vis)
    folium.raster_layers.TileLayer(
        tiles=mid['tile_fetcher'].url_format,
        attr='© GEE / CORINE / USGS', name=name,
        overlay=True, control=True, show=shown, opacity=opacity,
    ).add_to(fmap)

corine_vis = {'min': 0, 'max': 5, 'palette': CLASS_PALETTE}

m = folium.Map(location=MAP_CENTRE, zoom_start=MAP_ZOOM, tiles='CartoDB dark_matter')
add_ee_layer(m, corine_labels, corine_vis, 'CORINE 2018 (remapped)')
folium.LayerControl(collapsed=False).add_to(m)

# Legend
legend_html = '<div style="position:fixed;bottom:30px;left:30px;z-index:999;background:white;padding:10px;border-radius:5px;font-size:12px;">'
for name, color in CLASS_COLORS.items():
    legend_html += f'<div><span style="background:{color};width:12px;height:12px;display:inline-block;margin-right:5px;"></span>{name}</div>'
legend_html += '</div>'
m.get_root().html.add_child(folium.Element(legend_html))

display(m)


## 2. Build Feature Stack & Sample Training Pixels

Stack the 2023 composite (13 features) with CORINE labels, then use GEE's
`stratifiedSample` to pull balanced training pixels across all 6 classes.


In [5]:
# Load 2023 composite with all indices
def make_composite(year):
    def mask_clouds(img):
        qa = img.select('QA_PIXEL')
        return img.updateMask(qa.bitwiseAnd(1<<3).eq(0).And(qa.bitwiseAnd(1<<4).eq(0)))
    def scale_sr(img):
        return img.addBands(img.select('SR_B.').multiply(0.0000275).add(-0.2), overwrite=True)

    col_id = ('LANDSAT/LC09/C02/T1_L2' if year >= 2022 else
              'LANDSAT/LC08/C02/T1_L2' if year >= 2013 else
              'LANDSAT/LE07/C02/T1_L2' if year >= 1999 else
              'LANDSAT/LT05/C02/T1_L2')
    b_in  = (['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'] if year >= 2013
             else ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    b_out = ['blue','green','red','nir','swir1','swir2']

    comp = (ee.ImageCollection(col_id)
            .filterBounds(aoi).filterDate(f'{year}-05-01', f'{year}-09-30')
            .filter(ee.Filter.lt('CLOUD_COVER', 20))
            .map(mask_clouds).map(scale_sr)
            .select(b_in, b_out).median().clip(aoi))
    return gee_add_indices(comp)

composite_2023 = make_composite(2023)
print('2023 composite ready, bands:', composite_2023.bandNames().getInfo())


2023 composite ready, bands: ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'ndwi', 'ndbi', 'savi', 'bsi', 'evi', 'mndwi']


In [ ]:
training_image = composite_2023.select(FEATURE_BANDS).addBands(corine_labels)

# getInfo() hard limit is 5000 features total.
# 750 per class x max 6 classes = 4500, safely under the limit.
SAMPLES_PER_CLASS = 750

print(f'Sampling {SAMPLES_PER_CLASS} pixels per class from {len(available_classes)} classes...')
print(f'Classes: {[CLASS_NAMES[i] for i in available_classes]}\n')

samples_gee = training_image.stratifiedSample(
    numPoints=SAMPLES_PER_CLASS,
    classBand='label',
    region=aoi,
    scale=100,
    classValues=available_classes,
    classPoints=[SAMPLES_PER_CLASS] * len(available_classes),
    seed=42,
    geometries=False,
)

raw = samples_gee.getInfo()
df = pd.DataFrame([f['properties'] for f in raw['features']])

print(f'Total samples: {len(df)}')
print('\nSamples per class:')
for i in available_classes:
    n = (df['label'] == i).sum()
    print(f'  {i} - {CLASS_NAMES[i]:<22}  {n:>5} pixels')


## 3. Train & Evaluate - scikit-learn Random Forest


In [ ]:
X = df[FEATURE_BANDS].values
y = df['label'].values.astype(int)

active_names = [CLASS_NAMES[i] for i in available_classes]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print(f'Train: {len(X_train)} pixels  |  Test: {len(X_test)} pixels')

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
clf.fit(X_train, y_train)
print('Training complete')

y_pred = clf.predict(X_test)
oa    = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f'\nOverall Accuracy : {oa:.3f}  ({oa*100:.1f}%)')
print(f'Kappa coefficient: {kappa:.3f}')
print('\nPer-class report:')
print(classification_report(y_test, y_pred, target_names=active_names, digits=3))


In [ ]:
# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Proportion')

ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
short = [n.split('/')[0] for n in CLASS_NAMES]
ax.set_xticklabels(short, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(short, fontsize=10)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix (normalised)\nOA = {oa:.3f}  |  κ = {kappa:.3f}', fontsize=13)

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        val = cm_norm[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if val > 0.5 else 'black', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_2023.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to data/processed/confusion_matrix_2023.png')


In [ ]:
# Feature importance plot
importances = clf.feature_importances_
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2D8C3C' if 'ndvi' in FEATURE_BANDS[i] or 'savi' in FEATURE_BANDS[i] or 'evi' in FEATURE_BANDS[i]
          else '#3B82F6' if 'ndwi' in FEATURE_BANDS[i] or 'mndwi' in FEATURE_BANDS[i]
          else '#E8443A' if 'ndbi' in FEATURE_BANDS[i] or 'bsi' in FEATURE_BANDS[i]
          else '#6B7280'
          for i in order]

ax.bar(range(len(order)), importances[order], color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(order)))
ax.set_xticklabels([FEATURE_BANDS[i].upper() for i in order], rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Importance', fontsize=11)
ax.set_title('Feature Importance - Random Forest Classifier', fontsize=13)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../data/processed/feature_importance_2023.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Classify the Full 2023 Image (GEE server-side)

Train a GEE Random Forest on the same samples and apply it to every pixel
in the 2023 composite. This runs entirely on Google's servers - no download needed.


In [ ]:
# Train GEE Random Forest on the same samples
gee_clf = ee.Classifier.smileRandomForest(
    numberOfTrees=200,
    seed=42,
).train(
    features=samples_gee,
    classProperty='label',
    inputProperties=FEATURE_BANDS,
)

# Classify 2023 composite - every pixel gets a class 0-5
classified_2023 = (
    composite_2023.select(FEATURE_BANDS)
    .classify(gee_clf)
    .rename('classification')
    .clip(aoi)
)

print('Classification complete')
print('Unique class values:', classified_2023.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi, scale=100, maxPixels=1e8
).getInfo())


In [ ]:
# The classified map - this is the big visual payoff
classified_vis = {'min': 0, 'max': 5, 'palette': CLASS_PALETTE}
true_colour_vis = {'bands': ['red','green','blue'], 'min': 0, 'max': 0.3, 'gamma': 1.4}

m2 = folium.Map(location=MAP_CENTRE, zoom_start=MAP_ZOOM, tiles='CartoDB dark_matter')
add_ee_layer(m2, composite_2023,  true_colour_vis, 'True colour 2023', shown=False, opacity=1.0)
add_ee_layer(m2, classified_2023, classified_vis,  'Land cover 2023 (classified)')
folium.LayerControl(collapsed=False).add_to(m2)

# Legend
legend_html = '<div style="position:fixed;bottom:30px;left:30px;z-index:999;background:white;padding:10px;border-radius:5px;font-size:12px;">'
legend_html += f'<b>Land Cover Classes</b><br>'
for name, color in CLASS_COLORS.items():
    legend_html += f'<div><span style="background:{color};width:12px;height:12px;display:inline-block;margin-right:5px;"></span>{name}</div>'
legend_html += '</div>'
m2.get_root().html.add_child(folium.Element(legend_html))

display(m2)


In [ ]:
# Save the sklearn model
import pathlib
models_dir = pathlib.Path('../models')
models_dir.mkdir(exist_ok=True)

model_path = models_dir / 'rf_classifier_v1.joblib'
joblib.dump(clf, model_path)
print(f'Model saved: {model_path}')
print(f'Features : {FEATURE_BANDS}')
print(f'Classes  : {CLASS_NAMES}')
print(f'OA       : {oa:.3f}')
print(f'Kappa    : {kappa:.3f}')


## 5. Next Steps

With a working classifier producing labelled maps, Phase 3 is:

**Notebook 04 - Change Detection**
1. Classify composites for every 5 years: 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2023
2. Build transition matrices between each pair of years
3. Compute area time series per class (km²)
4. Identify the biggest transitions (scrubland → urban, volcanic → ???)
5. Validate against known events (resort expansion timelines, ISTAC population data)

---
*Labels: CORINE Land Cover 2018 (Copernicus/EEA)*  
*Classifier: Random Forest, 200 trees, 13 spectral features*  
*Model saved to `models/rf_classifier_v1.joblib`*
